In [ ]:
def main(datasources, start_date, end_date):
    import numpy as np
    import pandas as pd
    import dai

    bar_table = datasources["bar1m"]

    lookback_days = 60
    query_start_date = pd.to_datetime(start_date) - pd.Timedelta(days=lookback_days)

    sql = f"""
    WITH pool AS (
        SELECT DISTINCT instrument
        FROM bigalpha_2026_instruments
        WHERE date BETWEEN '{start_date}' AND '{end_date}'
    )
    SELECT
        date,
        instrument,
        close
    FROM {bar_table}
    PRUNE JOIN pool USING (instrument)
    """
    bar_df = dai.query(
        sql,
        filters={"date": [query_start_date, end_date]},
    ).df()

    if bar_df.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    bar_df["datetime"] = pd.to_datetime(bar_df["date"])
    bar_df["date"] = bar_df["datetime"].dt.normalize()
    bar_df = bar_df.sort_values(["instrument", "datetime"])

    daily_df = (
        bar_df.groupby(["date", "instrument"], as_index=False)
        .agg(close=("close", "last"))
        .sort_values(["instrument", "date"])
        .reset_index(drop=True)
    )

    daily_df["close"] = pd.to_numeric(daily_df["close"], errors="coerce")

    def _ts_zscore(series: pd.Series, window: int) -> pd.Series:
        mean = series.rolling(window, min_periods=min(2, window)).mean()
        std = series.rolling(window, min_periods=min(2, window)).std(ddof=1).replace(0, np.nan)
        return (series - mean) / std

    def calc_factor(group: pd.DataFrame) -> pd.DataFrame:
        group = group.sort_values("date").copy()
        delta_close_1 = group["close"].diff(1)
        zscore_delta_20 = _ts_zscore(delta_close_1, 20)
        raw = np.maximum(delta_close_1.to_numpy(), zscore_delta_20.to_numpy())
        raw = pd.Series(raw, index=group.index)
        smoothed = raw.rolling(3, min_periods=2).mean()
        group["factor"] = -pd.to_numeric(smoothed, errors="coerce")
        return group[["date", "instrument", "factor"]]

    factor_df = (
        daily_df.groupby("instrument", group_keys=False)
        .apply(calc_factor)
        .reset_index(drop=True)
    )

    factor_df["factor"] = (
        pd.to_numeric(factor_df["factor"], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )

    start_ts = pd.to_datetime(start_date).normalize()
    end_ts = pd.to_datetime(end_date).normalize()
    factor_df = factor_df[(factor_df["date"] >= start_ts) & (factor_df["date"] <= end_ts)].copy()

    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"]).dt.normalize()

    df = pd.merge(
        stk_pool,
        factor_df,
        how="left",
        on=["date", "instrument"],
    )

    df["factor"] = (
        pd.to_numeric(df["factor"], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )
    df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    df["instrument"] = df["instrument"].astype(str)

    return (
        df[["date", "instrument", "factor"]]
        .drop_duplicates(subset=["date", "instrument"], keep="last")
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )

